# 🏥 Chronic Kidney Disease (CKD) Progression & Risk Prediction System

## Complete Production-Ready ML/AI Project

This notebook contains the complete implementation of a CKD prediction system including:
- **Classification**: Predict CKD (Yes/No)
- **Regression**: Predict Kidney Function Score
- **Neural Networks**: MLP for both tasks
- **Explainability**: SHAP values and feature importance
- **AI Assistant**: Natural language explanations

---

**Author**: AI/ML Engineering Team  
**Version**: 1.0.0

---

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages (run this cell first in Colab)
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost shap plotly

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import warnings

# Scikit-learn imports
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc,
    mean_squared_error, mean_absolute_error, r2_score
)

# XGBoost
try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost loaded successfully")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available")

# SHAP for explainability
try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP loaded successfully")
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP not available")

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n✅ All libraries imported successfully!")

## 📊 Step 2: Create Dataset

We'll create a comprehensive CKD dataset with 100 patient records.

In [ ]:
# Create the CKD dataset
csv_data = """patient_id,age,gender,blood_pressure_systolic,blood_pressure_diastolic,blood_glucose,serum_creatinine,blood_urea_nitrogen,hemoglobin,albumin,specific_gravity,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,pedal_edema,anemia,smoking,family_history_ckd,bmi,kidney_function_score,ckd
1,48,Male,130,80,105,1.2,15,14.5,4.2,1.020,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.5,85.2,No
2,62,Female,150,95,180,2.8,45,9.5,3.1,1.010,abnormal,abnormal,present,present,Yes,Yes,No,poor,Yes,Yes,No,Yes,28.3,42.1,Yes
3,55,Male,145,90,145,1.8,28,12.0,3.8,1.015,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,No,26.7,68.5,No
4,70,Female,160,100,210,3.5,55,8.8,2.9,1.008,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,31.2,28.4,Yes
5,45,Male,120,75,95,1.0,12,15.2,4.5,1.025,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.8,92.3,No
6,58,Female,140,88,165,2.2,35,10.8,3.5,1.012,normal,abnormal,not_present,not_present,Yes,Yes,No,good,Yes,No,No,No,27.5,55.8,Yes
7,67,Male,155,92,190,2.9,48,9.2,3.0,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,Yes,Yes,29.8,38.2,Yes
8,42,Female,115,72,88,0.9,10,13.8,4.3,1.022,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.5,95.6,No
9,52,Male,135,85,125,1.5,22,13.2,4.0,1.018,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,25.2,78.4,No
10,73,Female,165,105,225,4.2,62,8.2,2.7,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,32.5,22.1,Yes
11,38,Male,118,74,92,0.95,11,15.0,4.4,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,23.1,94.2,No
12,65,Female,148,93,175,2.5,40,10.2,3.3,1.011,abnormal,abnormal,not_present,not_present,Yes,Yes,No,poor,Yes,No,No,Yes,28.9,48.5,Yes
13,50,Male,132,82,115,1.3,18,14.0,4.1,1.019,normal,normal,not_present,not_present,Yes,No,No,good,No,No,Yes,No,24.8,82.7,No
14,69,Female,158,98,205,3.2,52,8.5,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.8,30.5,Yes
15,44,Male,122,76,98,1.05,13,14.8,4.4,1.023,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.4,91.8,No
16,60,Female,152,94,185,2.6,42,9.8,3.2,1.010,abnormal,abnormal,present,not_present,Yes,Yes,No,poor,Yes,Yes,No,No,29.2,45.2,Yes
17,56,Male,142,89,155,1.9,30,11.5,3.7,1.014,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,Yes,26.3,62.8,No
18,75,Female,168,108,235,4.5,68,7.8,2.5,1.005,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,33.1,18.5,Yes
19,40,Male,116,73,90,0.92,10,15.3,4.5,1.025,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.8,96.2,No
20,63,Female,154,96,195,2.7,44,9.5,3.1,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.5,40.8,Yes
21,47,Male,128,79,108,1.15,14,14.2,4.2,1.020,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,24.2,86.5,No
22,71,Female,162,102,218,3.8,58,8.0,2.6,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,31.8,25.2,Yes
23,54,Male,138,86,138,1.7,26,12.5,3.9,1.016,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,No,25.9,71.2,No
24,66,Female,156,97,198,3.0,50,9.0,2.9,1.008,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.2,35.8,Yes
25,43,Male,120,75,95,1.0,12,15.0,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.6,93.5,No
26,59,Female,146,91,170,2.4,38,10.5,3.4,1.012,abnormal,abnormal,not_present,not_present,Yes,Yes,No,poor,Yes,No,No,No,28.1,52.5,Yes
27,51,Male,134,84,122,1.4,20,13.5,4.0,1.018,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,25.0,80.2,No
28,74,Female,166,106,228,4.3,65,7.9,2.5,1.005,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,32.8,20.5,Yes
29,39,Male,117,73,91,0.93,10,15.2,4.5,1.025,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.9,95.8,No
30,64,Female,153,95,192,2.8,46,9.4,3.0,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.8,42.5,Yes
31,49,Male,130,81,112,1.25,16,14.3,4.2,1.020,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.6,84.8,No
32,68,Female,159,99,208,3.3,54,8.6,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.5,32.1,Yes
33,46,Male,125,77,102,1.08,13,14.6,4.3,1.022,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,23.4,89.5,No
34,61,Female,151,94,182,2.5,41,10.0,3.2,1.010,abnormal,abnormal,present,not_present,Yes,Yes,No,poor,Yes,Yes,No,No,28.6,47.8,Yes
35,53,Male,136,85,132,1.6,24,12.8,3.9,1.017,normal,normal,not_present,not_present,Yes,No,No,good,No,No,Yes,No,25.5,75.5,No
36,76,Female,170,110,240,4.8,72,7.5,2.4,1.004,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,33.5,15.2,Yes
37,41,Male,118,74,93,0.96,11,15.1,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.0,94.8,No
38,57,Female,144,90,162,2.3,36,10.6,3.5,1.013,abnormal,abnormal,not_present,not_present,Yes,Yes,No,good,Yes,No,No,No,27.8,58.2,Yes
39,72,Male,164,104,222,4.0,60,8.3,2.6,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,Yes,Yes,31.5,24.5,Yes
40,37,Female,114,71,86,0.88,9,14.0,4.4,1.023,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,20.8,97.2,No
41,52,Male,133,83,118,1.35,19,13.8,4.1,1.019,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,25.1,81.5,No
42,67,Female,157,98,202,3.1,51,8.8,2.9,1.008,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.0,34.2,Yes
43,48,Male,129,80,110,1.2,15,14.4,4.2,1.020,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.4,85.8,No
44,63,Female,154,96,188,2.65,43,9.6,3.1,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.4,43.5,Yes
45,45,Male,122,76,98,1.02,12,14.9,4.4,1.023,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.7,92.0,No
46,70,Female,161,101,215,3.6,56,8.2,2.7,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,31.0,27.8,Yes
47,55,Male,140,88,148,1.85,29,11.8,3.8,1.015,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,No,26.5,66.8,No
48,78,Female,172,112,245,5.0,75,7.2,2.3,1.004,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,34.0,12.5,Yes
49,36,Male,115,72,88,0.9,9,15.4,4.6,1.026,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.2,97.8,No
50,60,Female,150,93,178,2.55,39,10.2,3.3,1.011,abnormal,abnormal,not_present,not_present,Yes,Yes,No,poor,Yes,No,No,No,28.4,50.2,Yes
51,50,Male,131,82,116,1.32,17,14.1,4.1,1.019,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.9,83.2,No
52,69,Female,160,100,212,3.4,55,8.4,2.7,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.6,29.5,Yes
53,44,Male,121,75,96,1.0,12,15.0,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.5,93.2,No
54,65,Female,155,97,195,2.85,47,9.3,3.0,1.008,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.6,39.8,Yes
55,47,Male,127,78,105,1.12,14,14.5,4.3,1.021,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,24.0,87.5,No
56,73,Female,167,107,232,4.4,66,7.8,2.5,1.005,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,32.6,19.2,Yes
57,54,Male,137,86,135,1.65,25,12.6,3.9,1.016,normal,normal,not_present,not_present,Yes,No,No,good,No,No,Yes,No,25.7,73.5,No
58,62,Female,152,95,185,2.62,42,9.7,3.2,1.010,abnormal,abnormal,present,not_present,Yes,Yes,No,poor,Yes,Yes,No,No,28.7,46.5,Yes
59,42,Male,119,74,94,0.98,11,15.2,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.2,95.2,No
60,66,Female,158,99,205,3.15,53,8.7,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.3,33.5,Yes
61,51,Male,134,84,120,1.38,20,13.6,4.0,1.018,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,25.2,79.8,No
62,75,Female,169,109,238,4.6,70,7.6,2.4,1.004,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,33.2,16.8,Yes
63,40,Male,117,73,91,0.94,10,15.3,4.5,1.025,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.7,96.0,No
64,64,Female,154,96,192,2.78,45,9.5,3.1,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.5,41.2,Yes
65,46,Male,124,77,100,1.05,13,14.7,4.4,1.022,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,23.2,90.2,No
66,71,Female,163,103,220,3.9,59,8.1,2.6,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,31.2,26.2,Yes
67,53,Male,135,85,128,1.55,23,13.0,4.0,1.017,normal,normal,not_present,not_present,Yes,No,No,good,No,No,Yes,No,25.4,76.8,No
68,68,Female,159,100,208,3.25,54,8.5,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.4,31.5,Yes
69,38,Male,116,72,89,0.92,10,15.2,4.5,1.025,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.5,95.5,No
70,58,Female,145,91,168,2.35,37,10.5,3.4,1.012,abnormal,abnormal,not_present,not_present,Yes,Yes,No,good,Yes,No,No,No,27.9,56.5,Yes
71,49,Male,130,81,114,1.28,17,14.2,4.2,1.020,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.7,84.2,No
72,77,Female,171,111,242,4.9,73,7.4,2.3,1.004,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,33.8,14.2,Yes
73,43,Male,120,75,95,1.0,12,15.0,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.4,93.8,No
74,61,Female,151,94,180,2.52,40,10.0,3.2,1.010,abnormal,abnormal,present,not_present,Yes,Yes,No,poor,Yes,Yes,No,No,28.5,48.2,Yes
75,56,Male,141,89,152,1.82,28,11.6,3.7,1.015,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,Yes,26.4,65.2,No
76,72,Male,165,105,225,4.1,62,8.2,2.6,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,Yes,Yes,31.6,23.8,Yes
77,35,Female,112,70,84,0.85,8,14.2,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,20.5,98.0,No
78,59,Female,147,92,172,2.42,38,10.4,3.4,1.012,abnormal,abnormal,not_present,not_present,Yes,Yes,No,poor,Yes,No,No,No,28.0,54.2,Yes
79,48,Male,128,79,108,1.18,15,14.4,4.2,1.020,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.3,86.2,No
80,67,Female,158,99,205,3.18,52,8.6,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.2,33.0,Yes
81,52,Male,133,83,122,1.42,21,13.5,4.0,1.018,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,25.1,80.5,No
82,74,Female,168,108,235,4.5,68,7.7,2.4,1.005,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,32.9,18.0,Yes
83,41,Male,118,74,92,0.95,11,15.1,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.0,94.5,No
84,63,Female,153,95,188,2.68,44,9.6,3.1,1.009,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.3,44.2,Yes
85,45,Male,123,76,99,1.04,13,14.8,4.4,1.023,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.8,91.5,No
86,69,Female,160,101,212,3.45,56,8.3,2.7,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.7,28.8,Yes
87,55,Male,139,87,145,1.78,27,12.0,3.8,1.015,normal,normal,not_present,not_present,Yes,Yes,No,good,No,No,Yes,No,26.2,68.2,No
88,79,Female,173,113,248,5.2,78,7.0,2.2,1.003,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,34.2,10.5,Yes
89,37,Male,115,72,88,0.9,9,15.4,4.6,1.026,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,21.3,97.5,No
90,60,Female,149,93,175,2.5,40,10.2,3.3,1.011,abnormal,abnormal,not_present,not_present,Yes,Yes,No,poor,Yes,No,No,No,28.3,51.5,Yes
91,50,Male,132,82,118,1.35,18,14.0,4.1,1.019,normal,normal,not_present,not_present,Yes,No,No,good,No,No,No,No,24.9,82.5,No
92,70,Female,162,102,218,3.7,58,8.1,2.6,1.006,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,31.0,27.0,Yes
93,44,Male,121,75,97,1.02,12,14.9,4.4,1.023,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.6,92.8,No
94,65,Female,156,97,198,2.92,48,9.2,3.0,1.008,abnormal,abnormal,present,not_present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,29.7,38.5,Yes
95,47,Male,126,78,104,1.1,14,14.5,4.3,1.021,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,23.8,88.2,No
96,76,Female,170,110,240,4.7,71,7.5,2.4,1.004,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,33.4,15.5,Yes
97,54,Male,138,86,138,1.68,26,12.5,3.9,1.016,normal,normal,not_present,not_present,Yes,No,No,good,No,No,Yes,No,25.8,72.5,No
98,62,Female,152,95,182,2.58,41,9.8,3.2,1.010,abnormal,abnormal,present,not_present,Yes,Yes,No,poor,Yes,Yes,No,No,28.6,47.2,Yes
99,42,Male,119,74,93,0.97,11,15.2,4.5,1.024,normal,normal,not_present,not_present,No,No,No,good,No,No,No,No,22.1,95.0,No
100,68,Female,159,100,210,3.35,55,8.5,2.8,1.007,abnormal,abnormal,present,present,Yes,Yes,Yes,poor,Yes,Yes,No,Yes,30.5,30.8,Yes"""

# Load data from string
df = pd.read_csv('kidney_disease.csv')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Dataset info
print("\n📊 Dataset Information:")
print("="*50)
df.info()

In [ ]:
# Statistical summary
df.describe()

## 📈 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CKD distribution (Classification Target)
ckd_counts = df['ckd'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(ckd_counts.values, labels=ckd_counts.index, autopct='%1.1f%%',
           colors=colors, explode=(0.05, 0.05), shadow=True, startangle=90)
axes[0].set_title('CKD Distribution\n(Classification Target)', fontsize=14, fontweight='bold')

# Kidney Function Score distribution (Regression Target)
axes[1].hist(df['kidney_function_score'], bins=25, color='#3498db', edgecolor='white', alpha=0.7)
axes[1].axvline(df['kidney_function_score'].mean(), color='#e74c3c', linestyle='--', linewidth=2, 
               label=f'Mean: {df["kidney_function_score"].mean():.1f}')
axes[1].axvline(df['kidney_function_score'].median(), color='#2ecc71', linestyle='--', linewidth=2,
               label=f'Median: {df["kidney_function_score"].median():.1f}')
axes[1].set_xlabel('Kidney Function Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Kidney Function Score Distribution\n(Regression Target)', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📊 CKD Distribution:")
print(df['ckd'].value_counts())

In [ ]:
# Numerical features distribution
numerical_cols = ['age', 'blood_pressure_systolic', 'blood_pressure_diastolic', 'blood_glucose',
                  'serum_creatinine', 'blood_urea_nitrogen', 'hemoglobin', 'albumin', 
                  'specific_gravity', 'bmi']

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    # Histogram with KDE
    df[col].hist(ax=ax, bins=20, color='#3498db', edgecolor='white', alpha=0.7)
    ax.axvline(df[col].mean(), color='#e74c3c', linestyle='--', linewidth=2)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('')

# Hide unused subplots
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots by CKD status
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    df.boxplot(column=col, by='ckd', ax=ax)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('CKD Status')

for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Features by CKD Status', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix
numerical_data = df.select_dtypes(include=[np.number]).drop(columns=['patient_id'])
corr_matrix = numerical_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
           cmap='RdBu_r', center=0, square=True,
           linewidths=0.5, cbar_kws={'shrink': 0.8},
           ax=ax, annot_kws={'size': 9})

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Risk Factors Analysis
risk_factors = ['hypertension', 'diabetes_mellitus', 'smoking', 'family_history_ckd']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, factor in enumerate(risk_factors):
    ct = pd.crosstab(df[factor], df['ckd'], normalize='index') * 100
    ckd_rates = ct['Yes'] if 'Yes' in ct.columns else pd.Series([0] * len(ct), index=ct.index)
    
    bars = axes[i].bar(ckd_rates.index, ckd_rates.values, 
                      color=['#2ecc71', '#e74c3c'] if len(ckd_rates) == 2 else ['#3498db'])
    axes[i].set_title(factor.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_ylabel('CKD Rate (%)')
    axes[i].set_ylim(0, 100)
    
    for bar in bars:
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{height:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('CKD Rate by Key Risk Factors', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔧 Step 4: Data Preprocessing

In [ ]:
# Define feature groups
numerical_features = [
    'age', 'blood_pressure_systolic', 'blood_pressure_diastolic',
    'blood_glucose', 'serum_creatinine', 'blood_urea_nitrogen',
    'hemoglobin', 'albumin', 'specific_gravity', 'bmi'
]

categorical_features = [
    'gender', 'red_blood_cells', 'pus_cell', 'pus_cell_clumps',
    'bacteria', 'hypertension', 'diabetes_mellitus',
    'coronary_artery_disease', 'appetite', 'pedal_edema',
    'anemia', 'smoking', 'family_history_ckd'
]

target_classification = 'ckd'
target_regression = 'kidney_function_score'

print(f"📊 Numerical features: {len(numerical_features)}")
print(f"📋 Categorical features: {len(categorical_features)}")

In [ ]:
def preprocess_data(df, for_classification=True):
    """
    Preprocess the CKD dataset.
    
    Args:
        df: Raw DataFrame
        for_classification: If True, prepare for classification; else regression
        
    Returns:
        X_train, X_test, y_train, y_test, feature_names, encoders, scaler
    """
    data = df.copy()
    
    # Drop patient_id
    if 'patient_id' in data.columns:
        data = data.drop('patient_id', axis=1)
    
    # Initialize encoders and scaler
    label_encoders = {}
    scaler = StandardScaler()
    
    # Encode categorical features
    for col in categorical_features:
        if col in data.columns:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col].astype(str))
            label_encoders[col] = le
    
    # Encode target for classification
    if for_classification:
        le_target = LabelEncoder()
        data[target_classification] = le_target.fit_transform(data[target_classification].astype(str))
        label_encoders[target_classification] = le_target
    
    # Prepare features and target
    feature_cols = [col for col in data.columns if col not in [target_classification, target_regression]]
    X = data[feature_cols]
    
    if for_classification:
        y = data[target_classification]
    else:
        y = data[target_regression]
    
    # Train-test split
    if for_classification:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE
        )
    
    # Scale numerical features
    num_cols = [col for col in numerical_features if col in X.columns]
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
    
    return (X_train_scaled.values, X_test_scaled.values, 
            y_train.values, y_test.values, 
            feature_cols, label_encoders, scaler)

In [ ]:
# Prepare Classification Data
X_train_cls, X_test_cls, y_train_cls, y_test_cls, feature_names, encoders_cls, scaler_cls = preprocess_data(df, for_classification=True)

print("\n📊 Classification Data:")
print(f"  Training: {X_train_cls.shape}")
print(f"  Test: {X_test_cls.shape}")
print(f"  Features: {len(feature_names)}")
print(f"  Class distribution (train): {np.bincount(y_train_cls)}")

In [ ]:
# Prepare Regression Data
X_train_reg, X_test_reg, y_train_reg, y_test_reg, _, encoders_reg, scaler_reg = preprocess_data(df, for_classification=False)

print("\n📊 Regression Data:")
print(f"  Training: {X_train_reg.shape}")
print(f"  Test: {X_test_reg.shape}")
print(f"  Target range: {y_train_reg.min():.1f} - {y_train_reg.max():.1f}")

## 🤖 Step 5: Classification Models

Training models to predict CKD (Yes/No)

In [ ]:
# Define classification models
classification_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, 
                                        random_state=RANDOM_STATE, early_stopping=True)
}

# Add XGBoost if available
if XGBOOST_AVAILABLE:
    classification_models['XGBoost'] = XGBClassifier(
        n_estimators=100, random_state=RANDOM_STATE, 
        use_label_encoder=False, eval_metric='logloss'
    )

print(f"\n🤖 Classification Models to Train: {list(classification_models.keys())}")

In [ ]:
# Train and evaluate classification models
classification_results = {}

print("\n" + "="*70)
print("TRAINING CLASSIFICATION MODELS")
print("="*70)

for name, model in classification_models.items():
    print(f"\n🔄 Training {name}...")
    
    # Train
    model.fit(X_train_cls, y_train_cls)
    
    # Predict
    y_pred = model.predict(X_test_cls)
    y_prob = model.predict_proba(X_test_cls)[:, 1]
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_test_cls, y_pred),
        'precision': precision_score(y_test_cls, y_pred, zero_division=0),
        'recall': recall_score(y_test_cls, y_pred, zero_division=0),
        'f1': f1_score(y_test_cls, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test_cls, y_prob),
        'y_pred': y_pred,
        'y_prob': y_prob,
        'model': model
    }
    
    classification_results[name] = metrics
    
    print(f"   ✅ Accuracy: {metrics['accuracy']:.4f}")
    print(f"   ✅ ROC-AUC: {metrics['roc_auc']:.4f}")

print("\n" + "="*70)
print("✅ All classification models trained successfully!")
print("="*70)

In [ ]:
# Classification Results Comparison
cls_comparison = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1'],
        'ROC-AUC': metrics['roc_auc']
    }
    for name, metrics in classification_results.items()
]).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("CLASSIFICATION MODEL COMPARISON")
print("="*70)
print(cls_comparison.to_string(index=False))

# Best model
best_cls_model = cls_comparison.iloc[0]['Model']
print(f"\n🏆 Best Classification Model: {best_cls_model}")

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set1(np.linspace(0, 1, len(classification_results)))

for (name, metrics), color in zip(classification_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test_cls, metrics['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Classification Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
n_models = len(classification_results)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (name, metrics) in enumerate(classification_results.items()):
    if i < len(axes):
        cm = confusion_matrix(y_test_cls, metrics['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['No CKD', 'CKD'],
                   yticklabels=['No CKD', 'CKD'],
                   ax=axes[i], annot_kws={'size': 14})
        axes[i].set_title(f'{name}\nAcc: {metrics["accuracy"]:.3f}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')

# Hide unused subplots
for j in range(len(classification_results), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices - Classification Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📈 Step 6: Regression Models

Training models to predict Kidney Function Score

In [ ]:
# Define regression models
regression_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso Regression': Lasso(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'MLP Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500,
                                       random_state=RANDOM_STATE, early_stopping=True)
}

# Add XGBoost if available
if XGBOOST_AVAILABLE:
    regression_models['XGBoost'] = XGBRegressor(
        n_estimators=100, random_state=RANDOM_STATE,
        objective='reg:squarederror'
    )

print(f"\n📈 Regression Models to Train: {list(regression_models.keys())}")

In [ ]:
# Train and evaluate regression models
regression_results = {}

print("\n" + "="*70)
print("TRAINING REGRESSION MODELS")
print("="*70)

for name, model in regression_models.items():
    print(f"\n🔄 Training {name}...")
    
    # Train
    model.fit(X_train_reg, y_train_reg)
    
    # Predict
    y_pred = model.predict(X_test_reg)
    
    # Calculate metrics
    mse = mean_squared_error(y_test_reg, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)
    
    # MAPE
    mask = y_test_reg != 0
    mape = np.mean(np.abs((y_test_reg[mask] - y_pred[mask]) / y_test_reg[mask])) * 100
    
    metrics = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'y_pred': y_pred,
        'model': model
    }
    
    regression_results[name] = metrics
    
    print(f"   ✅ R²: {r2:.4f}")
    print(f"   ✅ RMSE: {rmse:.4f}")

print("\n" + "="*70)
print("✅ All regression models trained successfully!")
print("="*70)

In [ ]:
# Regression Results Comparison
reg_comparison = pd.DataFrame([
    {
        'Model': name,
        'RMSE': metrics['rmse'],
        'MAE': metrics['mae'],
        'R²': metrics['r2'],
        'MAPE (%)': metrics['mape']
    }
    for name, metrics in regression_results.items()
]).sort_values('R²', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("REGRESSION MODEL COMPARISON")
print("="*70)
print(reg_comparison.to_string(index=False))

# Best model
best_reg_model = reg_comparison.iloc[0]['Model']
print(f"\n🏆 Best Regression Model: {best_reg_model}")

In [ ]:
# Actual vs Predicted plots
n_models = len(regression_results)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten()

for i, (name, metrics) in enumerate(regression_results.items()):
    ax = axes[i]
    ax.scatter(y_test_reg, metrics['y_pred'], alpha=0.6, c='#3498db')
    
    # Perfect prediction line
    min_val = min(y_test_reg.min(), metrics['y_pred'].min())
    max_val = max(y_test_reg.max(), metrics['y_pred'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name}\nR² = {metrics["r2"]:.3f}', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for j in range(len(regression_results), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Actual vs Predicted - Regression Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📊 Step 7: Model Comparison Dashboard

In [ ]:
# Summary Dashboard
fig = plt.figure(figsize=(16, 10))

# Classification ROC-AUC
ax1 = fig.add_subplot(2, 2, 1)
colors = plt.cm.RdYlGn(cls_comparison['ROC-AUC'])
bars = ax1.barh(cls_comparison['Model'], cls_comparison['ROC-AUC'], color=colors, alpha=0.8)
ax1.set_xlabel('ROC-AUC', fontsize=11)
ax1.set_title('Classification: ROC-AUC Scores', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 1.1)
ax1.grid(True, alpha=0.3, axis='x')
for bar in bars:
    ax1.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=10)

# Classification F1
ax2 = fig.add_subplot(2, 2, 2)
colors = plt.cm.RdYlGn(cls_comparison['F1-Score'])
bars = ax2.barh(cls_comparison['Model'], cls_comparison['F1-Score'], color=colors, alpha=0.8)
ax2.set_xlabel('F1-Score', fontsize=11)
ax2.set_title('Classification: F1 Scores', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1.1)
ax2.grid(True, alpha=0.3, axis='x')
for bar in bars:
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=10)

# Regression R²
ax3 = fig.add_subplot(2, 2, 3)
colors = plt.cm.RdYlGn(reg_comparison['R²'])
bars = ax3.barh(reg_comparison['Model'], reg_comparison['R²'], color=colors, alpha=0.8)
ax3.set_xlabel('R²', fontsize=11)
ax3.set_title('Regression: R² Scores', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 1.1)
ax3.grid(True, alpha=0.3, axis='x')
for bar in bars:
    ax3.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=10)

# Regression RMSE
ax4 = fig.add_subplot(2, 2, 4)
colors = plt.cm.RdYlGn_r(reg_comparison['RMSE'] / reg_comparison['RMSE'].max())
bars = ax4.barh(reg_comparison['Model'], reg_comparison['RMSE'], color=colors, alpha=0.8)
ax4.set_xlabel('RMSE', fontsize=11)
ax4.set_title('Regression: RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')
for bar in bars:
    ax4.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.2f}', va='center', fontsize=10)

fig.suptitle('CKD Prediction - Model Performance Summary', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 Step 8: Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
rf_classifier = classification_results['Random Forest']['model']
rf_regressor = regression_results['Random Forest']['model']

# Classification feature importance
cls_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

# Regression feature importance  
reg_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_regressor.feature_importances_
}).sort_values('importance', ascending=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Classification
top_cls = cls_importance.head(15)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_cls)))
bars = axes[0].barh(range(len(top_cls)), top_cls['importance'], color=colors, alpha=0.8)
axes[0].set_yticks(range(len(top_cls)))
axes[0].set_yticklabels(top_cls['feature'].apply(lambda x: x.replace('_', ' ').title()))
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Classification Feature Importance\n(Random Forest)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Regression
top_reg = reg_importance.head(15)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_reg)))
bars = axes[1].barh(range(len(top_reg)), top_reg['importance'], color=colors, alpha=0.8)
axes[1].set_yticks(range(len(top_reg)))
axes[1].set_yticklabels(top_reg['feature'].apply(lambda x: x.replace('_', ' ').title()))
axes[1].invert_yaxis()
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('Regression Feature Importance\n(Random Forest)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n📊 Top 10 Features for CKD Classification:")
print(cls_importance.head(10).to_string(index=False))

## 🔬 Step 9: SHAP Explainability (if available)

In [ ]:
if SHAP_AVAILABLE:
    print("🔬 Generating SHAP explanations...")
    
    # Create SHAP explainer for Random Forest classifier
    explainer = shap.TreeExplainer(rf_classifier)
    shap_values = explainer.shap_values(X_test_cls)
    
    # Summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values, 
                     X_test_cls, feature_names=feature_names, show=False)
    plt.title('SHAP Feature Importance (Classification)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ SHAP not available. Skipping SHAP analysis.")
    print("   Install with: !pip install shap")

## 🤖 Step 10: AI Assistant - Prediction Explanations

In [ ]:
class CKDAssistant:
    """AI Assistant for CKD risk explanations."""
    
    def __init__(self):
        self.risk_thresholds = {'low': 0.3, 'moderate': 0.7, 'high': 1.0}
    
    def interpret_risk(self, probability):
        """Interpret CKD risk probability."""
        if probability < self.risk_thresholds['low']:
            return {
                'level': 'LOW',
                'color': 'green',
                'description': 'Low risk of CKD. Continue healthy lifestyle.',
                'recommendations': [
                    'Maintain regular health check-ups',
                    'Continue balanced diet',
                    'Stay physically active'
                ]
            }
        elif probability < self.risk_thresholds['moderate']:
            return {
                'level': 'MODERATE', 
                'color': 'yellow',
                'description': 'Moderate risk. Monitor and address risk factors.',
                'recommendations': [
                    'Schedule consultation with healthcare provider',
                    'Get kidney function tests',
                    'Control blood pressure and blood sugar',
                    'Review current medications'
                ]
            }
        else:
            return {
                'level': 'HIGH',
                'color': 'red', 
                'description': 'High risk of CKD. Immediate attention needed.',
                'recommendations': [
                    'URGENT: Consult nephrologist immediately',
                    'Comprehensive kidney function tests',
                    'Strict blood pressure and glucose control',
                    'Follow kidney-friendly diet',
                    'Regular monitoring required'
                ]
            }
    
    def interpret_kidney_score(self, score):
        """Interpret kidney function score."""
        if score >= 90:
            return {'status': 'NORMAL', 'stage': 'Stage 1', 'description': 'Normal kidney function'}
        elif score >= 60:
            return {'status': 'MILDLY REDUCED', 'stage': 'Stage 2', 'description': 'Mildly decreased function'}
        elif score >= 30:
            return {'status': 'MODERATELY REDUCED', 'stage': 'Stage 3', 'description': 'Moderate decrease'}
        elif score >= 15:
            return {'status': 'SEVERELY REDUCED', 'stage': 'Stage 4', 'description': 'Severe decrease'}
        else:
            return {'status': 'KIDNEY FAILURE', 'stage': 'Stage 5', 'description': 'Kidney failure'}
    
    def generate_report(self, patient_data, ckd_prob, kidney_score):
        """Generate patient report."""
        risk = self.interpret_risk(ckd_prob)
        kidney = self.interpret_kidney_score(kidney_score)
        
        report = f"""
╔══════════════════════════════════════════════════════════════╗
║                 CKD RISK ASSESSMENT REPORT                   ║
╠══════════════════════════════════════════════════════════════╣
║  CKD Risk Probability: {ckd_prob*100:5.1f}%                              ║
║  Risk Level: {risk['level']:<10}                                    ║
╠══════════════════════════════════════════════════════════════╣
║  Kidney Function Score: {kidney_score:5.1f}                             ║
║  Status: {kidney['status']:<15}                              ║
║  Stage: {kidney['stage']:<10}                                       ║
╚══════════════════════════════════════════════════════════════╝

ASSESSMENT: {risk['description']}

RECOMMENDATIONS:
"""
        for i, rec in enumerate(risk['recommendations'], 1):
            report += f"  {i}. {rec}\n"
        
        report += "\n" + "="*60
        report += "\n⚠️ DISCLAIMER: This is AI-generated and not medical advice."
        report += "\n   Consult healthcare professionals for proper diagnosis."
        
        return report

# Create assistant
assistant = CKDAssistant()
print("✅ AI Assistant initialized!")

In [ ]:
# Demo: Make predictions for sample patients
print("\n" + "="*70)
print("AI ASSISTANT - SAMPLE PATIENT PREDICTIONS")
print("="*70)

# Get best models
best_cls = classification_results[best_cls_model]['model']
best_reg = regression_results[best_reg_model]['model']

# Test on a few samples
for i in range(3):
    # Get prediction
    ckd_prob = best_cls.predict_proba(X_test_cls[i:i+1])[0][1]
    kidney_score = best_reg.predict(X_test_reg[i:i+1])[0]
    
    # Generate report
    sample_patient = {'age': df.iloc[i]['age'], 'gender': df.iloc[i]['gender']}
    report = assistant.generate_report(sample_patient, ckd_prob, kidney_score)
    
    print(f"\n--- Patient {i+1} ---")
    print(report)

## 🎯 Step 11: Interactive Prediction Function

In [ ]:
def predict_ckd_risk(patient_data, classifier, regressor, scaler, encoders, feature_names):
    """
    Make CKD prediction for a single patient.
    
    Args:
        patient_data: Dictionary with patient features
        classifier: Trained classification model
        regressor: Trained regression model
        scaler: Fitted StandardScaler
        encoders: Fitted LabelEncoders
        feature_names: List of feature names
    
    Returns:
        Dictionary with predictions and interpretation
    """
    # Create DataFrame
    df_patient = pd.DataFrame([patient_data])
    
    # Encode categorical features
    for col in categorical_features:
        if col in df_patient.columns and col in encoders:
            # Handle unseen labels
            known_labels = set(encoders[col].classes_)
            df_patient[col] = df_patient[col].apply(
                lambda x: x if x in known_labels else encoders[col].classes_[0]
            )
            df_patient[col] = encoders[col].transform(df_patient[col].astype(str))
    
    # Scale numerical features
    num_cols = [col for col in numerical_features if col in df_patient.columns]
    df_patient[num_cols] = scaler.transform(df_patient[num_cols])
    
    # Get features in correct order
    X = df_patient[feature_names].values
    
    # Make predictions
    ckd_prob = classifier.predict_proba(X)[0][1]
    kidney_score = regressor.predict(X)[0]
    
    # Get interpretations
    assistant = CKDAssistant()
    risk_interp = assistant.interpret_risk(ckd_prob)
    kidney_interp = assistant.interpret_kidney_score(kidney_score)
    
    return {
        'ckd_probability': ckd_prob,
        'ckd_risk_level': risk_interp['level'],
        'kidney_function_score': kidney_score,
        'kidney_status': kidney_interp['status'],
        'recommendations': risk_interp['recommendations'],
        'report': assistant.generate_report(patient_data, ckd_prob, kidney_score)
    }

print("✅ Prediction function created!")

In [ ]:
# Test with a sample patient
sample_patient = {
    'age': 58,
    'gender': 'Male',
    'blood_pressure_systolic': 145,
    'blood_pressure_diastolic': 92,
    'blood_glucose': 165,
    'serum_creatinine': 1.8,
    'blood_urea_nitrogen': 28,
    'hemoglobin': 11.5,
    'albumin': 3.5,
    'specific_gravity': 1.015,
    'red_blood_cells': 'normal',
    'pus_cell': 'normal',
    'pus_cell_clumps': 'not_present',
    'bacteria': 'not_present',
    'hypertension': 'Yes',
    'diabetes_mellitus': 'Yes',
    'coronary_artery_disease': 'No',
    'appetite': 'good',
    'pedal_edema': 'No',
    'anemia': 'No',
    'smoking': 'No',
    'family_history_ckd': 'Yes',
    'bmi': 28.5
}

# Make prediction
result = predict_ckd_risk(
    sample_patient,
    classification_results[best_cls_model]['model'],
    regression_results[best_reg_model]['model'],
    scaler_cls,
    encoders_cls,
    feature_names
)

print("\n" + "="*70)
print("SAMPLE PATIENT PREDICTION")
print("="*70)
print(result['report'])

## 📋 Step 12: Summary & Conclusions

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         CKD PREDICTION SYSTEM - PROJECT SUMMARY                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 DATASET                                                          ║
║  • 100 patient records with 25 features                              ║
║  • Classification target: CKD (Yes/No)                               ║
║  • Regression target: Kidney Function Score                          ║
║                                                                      ║
║  🤖 CLASSIFICATION MODELS                                            ║
║  • Logistic Regression                                               ║
║  • Random Forest                                                     ║
║  • XGBoost (if available)                                            ║
║  • Gradient Boosting                                                 ║
║  • Neural Network (MLP)                                              ║
║                                                                      ║
║  📈 REGRESSION MODELS                                                ║
║  • Linear Regression                                                 ║
║  • Ridge & Lasso Regression                                          ║
║  • Random Forest                                                     ║
║  • Gradient Boosting                                                 ║
║  • XGBoost (if available)                                            ║
║  • Neural Network (MLP)                                              ║
║                                                                      ║
║  🔍 EXPLAINABILITY                                                   ║
║  • Feature Importance Analysis                                       ║
║  • SHAP Values (if available)                                        ║
║                                                                      ║
║  🤖 AI ASSISTANT                                                     ║
║  • Risk Level Interpretation                                         ║
║  • Kidney Function Status                                            ║
║  • Personalized Recommendations                                      ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

print("\n📊 FINAL RESULTS:")
print("="*60)
print(f"\n🏆 Best Classification Model: {best_cls_model}")
best_cls_metrics = classification_results[best_cls_model]
print(f"   • Accuracy: {best_cls_metrics['accuracy']:.4f}")
print(f"   • ROC-AUC: {best_cls_metrics['roc_auc']:.4f}")
print(f"   • F1-Score: {best_cls_metrics['f1']:.4f}")

print(f"\n🏆 Best Regression Model: {best_reg_model}")
best_reg_metrics = regression_results[best_reg_model]
print(f"   • R²: {best_reg_metrics['r2']:.4f}")
print(f"   • RMSE: {best_reg_metrics['rmse']:.4f}")
print(f"   • MAE: {best_reg_metrics['mae']:.4f}")

print("\n" + "="*60)
print("✅ Project completed successfully!")
print("="*60)

---

## 🎉 Congratulations!

You have successfully completed the CKD Prediction System project!

### What you've learned:
1. **Data Preprocessing**: Handling missing values, encoding, scaling
2. **EDA**: Visualizing distributions, correlations, risk factors
3. **Classification**: Multiple models for CKD prediction
4. **Regression**: Kidney function score prediction
5. **Neural Networks**: MLP for both tasks
6. **Model Evaluation**: ROC curves, confusion matrices, metrics comparison
7. **Explainability**: Feature importance, SHAP values
8. **AI Integration**: Natural language explanations

### Next Steps:
- Add more data for better model performance
- Try hyperparameter tuning
- Deploy as a web application with Streamlit
- Add more advanced deep learning models

---

**Made with ❤️ for better kidney health prediction**